In [3]:
import os
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoConfig
from torch.optim import AdamW
from sklearn.metrics import f1_score
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


In [4]:
current_dir = os.getcwd()
split_dir = os.path.join(current_dir, "centralized_split")

print("Notebook dir:", current_dir)
print("Split dir   :", split_dir)

df_train = pd.read_pickle(os.path.join(split_dir, "train.pkl"))
df_val   = pd.read_pickle(os.path.join(split_dir, "val.pkl"))

print(df_train.shape, df_val.shape)
df_train.head()


Notebook dir: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi
Split dir   : c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi\centralized_split
(9017, 14) (1932, 14)


,input_ids,attention_mask,HS,Abusive,HS_Individual,HS_Group,HS_Religion,HS_Race,HS_Physical,HS_Gender,HS_Other,HS_Weak,HS_Moderate,HS_Strong
0,"[2, 5505, 2956, 4299, 30470, 5505, 4299, 186, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,1,0,0,0,0,0,0,0,0,0,0
1,"[2, 9283, 30470, 30470, 2589, 9479, 2409, 6690...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,0,0,0,0,0,0,0,0
2,"[2, 977, 1028, 30361, 5165, 6013, 17722, 977, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1,0,0,1,0,0,0,0,1,0,0,1
3,"[2, 379, 1626, 30371, 8176, 4911, 761, 8812, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,0,0,0,0,0,0,0,0
4,"[2, 2120, 28731, 225, 5157, 17, 13987, 30468, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1,1,1,0,0,0,0,0,1,1,0,0


In [5]:
feature_cols = ["input_ids", "attention_mask"]
label_cols = [c for c in df_train.columns if c not in feature_cols]

print("Label columns:", label_cols)
print("Jumlah label :", len(label_cols))


Label columns: ['HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong']
Jumlah label : 12


In [6]:
class HateSpeechDataset(Dataset):
    def __init__(self, df, label_cols):
        self.input_ids = df["input_ids"].tolist()
        self.attention_mask = df["attention_mask"].tolist()
        self.labels = df[label_cols].values.astype("float32")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }

train_dataset = HateSpeechDataset(df_train, label_cols)
val_dataset   = HateSpeechDataset(df_val,   label_cols)

BATCH_SIZE = 16   # kalau RAM kuat bisa naik ke 12 / 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(val_loader)


(564, 121)

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
NUM_LABELS = len(label_cols)

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = NUM_LABELS
config.problem_type = "multi_label_classification"

base_model = AutoModel.from_pretrained(MODEL_NAME)

class IndoBertForMultiLabel(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.bert = base_model
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0]  # ambil [CLS]
        cls_token = self.dropout(cls_token)
        logits = self.classifier(cls_token)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels)

        return loss, logits

model = IndoBertForMultiLabel(base_model, NUM_LABELS).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=2e-5)

print("Model ready on", DEVICE)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

c:\Users\Alif\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alif\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model ready on cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [8]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Train", leave=False):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        loss, logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss


def eval_one_epoch(model, dataloader, device, threshold=0.5):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_preds  = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Val", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss, logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > threshold).long()

            all_labels.append(labels.cpu())
            all_preds.append(preds.cpu())

    avg_loss = total_loss / len(dataloader)

    all_labels = torch.cat(all_labels, dim=0).numpy()
    all_preds  = torch.cat(all_preds,  dim=0).numpy()

    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, micro_f1, macro_f1


In [9]:
EPOCHS = 1   # kalau masih kuat, nanti bisa tambah

best_val_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    print(f"\n====== EPOCH {epoch}/{EPOCHS} ======")

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, val_micro_f1, val_macro_f1 = eval_one_epoch(model, val_loader, DEVICE)

    print(f"Train loss : {train_loss:.4f}")
    print(f"Val loss   : {val_loss:.4f}")
    print(f"Val microF1: {val_micro_f1:.4f}")
    print(f"Val macroF1: {val_macro_f1:.4f}")



====== EPOCH 1/1 ======


Train:   0%|          | 0/564 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Train loss : 0.2568
Val loss   : 0.1998
Val microF1: 0.7250
Val macroF1: 0.5531


In [10]:
models_dir = os.path.join(current_dir, "models")
os.makedirs(models_dir, exist_ok=True)

warmstart_path = os.path.join(models_dir, "model_warmstart.pt")
torch.save(model.state_dict(), warmstart_path)

print("Saved warm-start model to:", warmstart_path)


Saved warm-start model to: c:\Users\Alif\OneDrive - uinjkt.ac.id\Kuliah UIN\Skripsi\eksperimen skripsi\models\model_warmstart.pt
